# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [3]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'https://edwarddonner.com/2024/10/16/from-software-engineer-to-ai-data-scientist-resources/',
 'https://edwarddonner.com/2024/10/16/from-software-engineer-to-ai-data-scientist-resources/',
 'https://edwarddonner.com/2024/08/06/outsmart/',
 'https://edwarddonner.com/2024/08/06/outsmart/',
 'https://edwarddonner.com/2024/06/26/choosing-the-right-llm-resources/

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [7]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [8]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2024/11/13/llm-engineering-resources/
https://edwarddonner.com/2024/11/13/llm-engineering-resources/
https://edwarddonner.com/2024/10/16/from-software-engineer-to-ai-data-scientist-resources/
https://edwarddonner

In [9]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [10]:
anthropic = Website("https://anthropic.com")
anthropic.links

[]

In [11]:
get_links("https://anthropic.com")

{'links': [{'type': 'about page', 'url': 'https://anthropic.com/about'},
  {'type': 'careers page', 'url': 'https://anthropic.com/careers'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [12]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [13]:
print(get_all_details("https://anthropic.com"))

Found links: {'links': [{'type': 'about page', 'url': 'https://anthropic.com/about'}, {'type': 'careers page', 'url': 'https://anthropic.com/careers'}]}
Landing page:
Webpage Title:
Home \ Anthropic
Webpage Contents:




about page
Webpage Title:
Not Found \ Anthropic
Webpage Contents:




careers page
Webpage Title:
Careers \ Anthropic
Webpage Contents:





In [21]:
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncate if more than 20,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("Anthropic", "https://anthropic.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://anthropic.com/about'}, {'type': 'careers page', 'url': 'https://anthropic.com/careers'}]}


'You are looking at a company called: Anthropic\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHome \\ Anthropic\nWebpage Contents:\n\n\n\n\nabout page\nWebpage Title:\nNot Found \\ Anthropic\nWebpage Contents:\n\n\n\n\ncareers page\nWebpage Title:\nCareers \\ Anthropic\nWebpage Contents:\n\n\n'

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("Anthropic", "https://anthropic.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://anthropic.com/about'}, {'type': 'careers page', 'url': 'https://anthropic.com/careers'}]}


# 🌟 Welcome to Anthropic: Where AI Dreams Come True! 🌟

## Who Are We?
At **Anthropic**, we’re like the friendly neighborhood superheroes, but instead of capes, we wear lab coats! Our mission? To build AI that cares about people—because who wouldn’t want a robot that remembers your birthday?

*Note: There is no about page… it's a bit of a mystery! Maybe we're too busy saving the world from rogue AI?*

---

## Our Culture: AI with a Side of Fun!
Here at Anthropic, our culture is all about **Respect, Responsibility, and a Dash of Quirkiness**. We believe in open communication, so we encourage debates about anything from the smartest algorithms to the best toppings on pizza (Pineapple believers, we see you!).

- **Teamwork**: Our meetings often resemble a friendly competition. "Who can launch the next algorithm fastest?" Spoiler alert: Everyone wins!
- **Innovation with a Twist**: We love a good brain-stretching session. Warning: may lead to frequent lightbulb moments (not liable for sudden bursts of genius).
- **Happiness is Key**: We uphold a strict tradition of ensuring everyone has a reason to smile—and donuts. Lots of glorious donuts!

---

## Our Customers: The Future is in Your Hands
Our clientele includes businesses from different sectors, confidants who want AI that acts like a pal rather than a pesky overlord. We’ve partnered with innovators who understand that humanity and technology can shake hands—and we’ve got the snacks to prove it!

---

## Careers at Anthropic: Join the Fun!
Want to embark on an exciting journey with us? Our **Careers Page** is where dreams become realities (just like in fairy tales, but with less drama). We’re on the lookout for:

- **AI Wizards**: If you can make a machine learn as easily as making toast, we've got a seat for you!
- **Philosophers of Technology**: For those who ponder the meaning of existence and how to not accidentally create Skynet.
- **Team Enthusiasts**: If you're known for your ability to bring people together with puns and snacks—yes, we want you!

---

## Connect with Us!
So, if you’re looking for a place where innovation meets the soul, where laughter is encouraged, and where you can help create intelligent assistants that don’t just compute but COMPASSIONATE-ize—**Anthropic is the place for you!**

*Disclaimer: No robots were harmed in the making of this brochure. Please practice safe coding!*

🐦 **Follow us on Twitter** | 🌐 **Visit our Website** | 💼 **Check out Job Openings**  

---

Remember: At Anthropic, we believe AI shouldn't just compute—*it should connect, care, and create lasting memories!*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("Anthropic", "https://anthropic.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://anthropic.com/about/'}, {'type': 'careers page', 'url': 'https://anthropic.com/careers/'}]}


# Welcome to the Wondrous World of Anthropic!

### Who Are We?
At Anthropic, we're about as mysterious as an unmarked package. Sure, we might have a few missing pages (like our about page — who knows where that went?), but what we do have is a team dedicated to building AI systems that are safe, reliable, and overall just plain fantastic!

### Our Culture: An Understatement of Awesomeness
Think of us as a quirky hive of brilliant minds—imagine a mad scientist convention crossed with a book club, where discussions include everything from machine learning to the perfect way to brew a cup of coffee! Here, collaboration is our jam, communication is the name of the game, and a little laughter goes a long way (and probably helps our AIs, too). 

We thrive on open communication and believe that every idea—from the utterly ridiculous to the genius-level brilliant—is valuable. After all, someone has to come up with the next AI that can fetch coffee!

### Who Do We Serve?  
We don’t just cater to tech geniuses and AI aficionados (but yes, they’re welcome too)! Our clientele ranges from curious startups to behemoth corporations, all yearning for that sprinkle of intelligence that only Anthropic can provide. So whether you're a small fish in a big pond or a shark in the corporate sea, we’ve got something up our sleeves (if only we could find them) to meet your needs!

### Join Us!  
Looking to dive into an ocean of opportunities? Our careers page is brimming with possibilities, all marked by a welcoming "Help Wanted" sign! Whether you're an AI whisperer or just someone who can tell the difference between a cat and a toaster, we’d love to hear from you.

At Anthropic, you'll work with some of the brightest minds, engage in groundbreaking projects, and hopefully learn to tell a joke or two (we promise our AIs are getting better at it). 

### Why Choose Anthropic?
- **Flexible Work Options**: Whether you prefer to turn your home office into a productivity palace or work from our overly-stimulating coffee shop nearby, your call! 
- **Innovative Environment**: You won't find any boring boardrooms here! If "think outside the box" were a team mantra, we’d be a bunch of box-dodgers!
- **Snack-Laden Office**: Forget the gym! Our office is stocked with snacks to fuel your intellectual snacking (and yes, we take our snack game very seriously).

### Join the Anthropic Experience!
Come for the coffee, stay for the mind-boggling discussions. Together, we’ll figure out this whole "working with AI" thing, one laugh at a time!

---

**Anthropic** – Because even AI needs a little humanity (and a side of humor)! 

*Disclaimer: Page contents may disappear, but our commitment to brilliance remains front and center.*

In [28]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Welcome to Hugging Face!

Where AI isn't just a buzzword, it hugs you warmly and offers you a nice digital cookie too! 🍪

---

## Who Are We?
At Hugging Face, we are on a **mission to democratize** awesome Machine Learning (ML) one commit at a time. We are not just building cutting-edge AI; we’re weaving a community tapestry of researchers, creators, and some of the most brilliant minds in the tech field all around the globe. Because sometimes, it’s not enough to just be brilliant—you have to be *snuggly* too! 🤗

---

## Our Culture
At Hugging Face, we’re like a cozy blanket fort made of innovation, collaboration, and a sprinkle of nerdy humor. Here’s what you can expect:

- **Collaboration**: Our platform thrives on teamwork. Whether you're sharing a model, dataset, or just your Sunday brunch pictures, everyone’s invited!
- **Open Source**: We believe in sharing—like that last piece of pizza at a party you know everyone will want. 🍕
- **Playfulness**: We love turning complex AI models into toys. Literally. With over 400k+ models browsable, you can play with them like they’re the coolest LEGO sets in the universe!

---

## Who's Using Us?
We’ve got over **50,000 organizations** chewing through our resources like it’s an all-you-can-eat buffet:

- **AI at Meta**
- **Amazon Web Services**
- **Microsoft** 
- **Google**
  
Just to name a few big fish in our pond. Whether you’re a startup in a garage or a tech giant, we cater to everyone with a dash of flair. 🎉

---

## Career Opportunities: Join the Fun!
Thinking of hopping on the Hugging Face train? 🚂 Here’s what you’ll be stepping into:

- **Innovative Minds**: Work alongside the brightest brains who aren’t just here for the paycheck—they’re here to change the world (and probably make some dad jokes).
- **Growth Opportunities**: Develop your skills on the go with hands-on projects and mentorship. Who knew the path to success was paved with *really cool AI tools*?
- **Flexible Work Environment**: Want to work in your pajamas while sipping coffee? No problem! Just don’t send us your PJs, please.

---

## Pricing: Spoiler Alert - You Won’t Break the Bank!
- **Forever Free**: Collaborate on ML projects without paying a dime. Yup, you read that right!
- **Pro Plans**: For only $9/month, unlock advanced features that make your model sing and dance (figuratively).
- **Enterprise**: Starting at just $20/user/month, get top-tier support, advanced security, and features for your organization!

---

## Why Hugging Face?
Because let’s face it (pun intended), we make AI feel a little less intimidating and a lot more friendly! You no longer have to hide from AI in fear—we're here to give it a hug. 🤗💖

---

### **Join Us!**
Ready to be part of a vibrant community tearing down the wall between humans and machines? Sign up today and let’s change the world together—one friendly hug at a time!

--- 

👕 Hugging Face: The AI community building the future—one model, one hug, and a dash of silliness at a time!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 2 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!
            </span>
        </td>
    </tr>
</table>